In [2]:
import pandas as pd
import numpy as np
# import pyreadstat
from pathlib import Path

In [ ]:

"""
Project: Demele-Jev (ANRS0629) Cambodia
Purpose: Data preparation & monitoring
Converted from Stata to Python

Author: Python pipeline version
"""

# ---------------------------------------------------------
# PATH CONFIGURATION
# ---------------------------------------------------------

BASE = Path("/Users/kennareyseang/Documents/Projects and Proposals/FEF:ANRS/Data collection/Data extracted/Study data")

screen1_file = BASE / "D1_Screening_Form_Cohort1_Baseline.xlsx"
screen2_file = BASE / "D4_Screening_Form_Cohort2_Baseline.xlsx"

lab_elisa_c1 = BASE / "C1_LabResult_Cohort1_Baseline_ELISA.xlsx"
lab_elisa_c2 = BASE / "C4_LabResult_Cohort2_Baseline_ELISA.xlsx"
lab_pcr_c2 = BASE / "C5_LabResult_Cohort2_Baseline_QIASTAT_PCR.xlsx"
lab_genx_c2 = BASE / "C6_LabResult_Cohort2_Baseline_ IPC_GenXpert.xlsx"


# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------

def load_excel(file, sheet, cols):
    df = pd.read_excel(
        file,
        sheet_name=sheet,
        skiprows=1,
        usecols=cols
    )
    df.columns = df.columns.str.lower()
    return df


def fix_ids(df, col_manual="a1manual"):
    """Replace a1 with manual ID if provided"""
    mask = df[col_manual].notna()
    df.loc[mask, "a1"] = df.loc[mask, col_manual]
    return df


def detect_duplicates(df, idcol):
    df = df.sort_values(idcol)
    df["dup"] = df.groupby(idcol).cumcount()+1
    df.loc[df.groupby(idcol)[idcol].transform("count")==1,"dup"] = 0
    return df


def save_dta(df, name):
    pyreadstat.write_dta(df, BASE / name)


# ---------------------------------------------------------
# 1. SCREENING DATA
# ---------------------------------------------------------

print("Loading screening data...")

screen1 = load_excel(
    screen1_file,
    "Sheet 1 - D1_Screening_Form_Coh",
    "A:AV"
)

screen2 = load_excel(
    screen2_file,
    "Sheet 1 - D4_Screening_Form_Coh",
    "A:AT"
)

screen2["a1manual"] = screen2["a1manual"].astype(str)

screen1 = fix_ids(screen1)
screen2 = fix_ids(screen2)

screen = pd.concat([screen1, screen2], ignore_index=True)


# ---------------------------------------------------------
# CREATE COHORT VARIABLE
# ---------------------------------------------------------

screen["cohort"] = np.nan

screen.loc[screen["a1"].str.contains("^DEM1", na=False), "cohort"] = 1
screen.loc[screen["a1"].str.contains("^DEM2", na=False), "cohort"] = 2

screen.loc[screen["a1manual"].str.contains("^DEM1", na=False), "cohort"] = 1
screen.loc[screen["a1manual"].str.contains("^DEM2", na=False), "cohort"] = 2


# ---------------------------------------------------------
# ADDRESS RECODING
# ---------------------------------------------------------

address_map = {f"KH-{i}":i for i in range(1,26)}

screen["address"] = screen["a6"].map(address_map)
screen["address"] = screen["address"].fillna(
    screen["scr_a6"].map(address_map)
)


# ---------------------------------------------------------
# SITE
# ---------------------------------------------------------

screen["site"] = np.nan
screen.loc[(screen.scr_a3==1)|(screen.a3==1),"site"] = 1
screen.loc[(screen.scr_a3==2)|(screen.a3==2),"site"] = 2


# ---------------------------------------------------------
# AGE CATEGORY
# ---------------------------------------------------------

screen["agecat"] = np.nan

screen.loc[(screen.scr_a4==1)|(screen.a4==1),"agecat"]=1
screen.loc[(screen.scr_a4==2)|(screen.a4==2),"agecat"]=2
screen.loc[(screen.scr_a4==3)|(screen.a4==3),"agecat"]=3


# ---------------------------------------------------------
# SEX
# ---------------------------------------------------------

screen["sex"]=np.nan
screen.loc[(screen.scr_a5==1)|(screen.a5==1),"sex"]=1
screen.loc[(screen.scr_a5==2)|(screen.a5==2),"sex"]=2


# ---------------------------------------------------------
# CONSENT STATUS
# ---------------------------------------------------------

screen["a7new"]=np.nan

screen.loc[
    (screen.scr_a72==1)&(screen.scr_a73==1)&(screen.scr_a74==1),
    "a7new"
]=1

screen.loc[
    (screen.a72==1)&(screen.a73==1)&(screen.a74==1),
    "a7new"
]=1

screen.loc[screen.scr_a7.astype(str).str.startswith("1",na=False),"a7new"]=1
screen.loc[screen.a7.astype(str).str.startswith("1",na=False),"a7new"]=1

screen.loc[
    (screen.scr_a72==1)&(screen.scr_a73==1)&(screen.scr_a74==0),
    "a7new"
]=2

screen.loc[
    (screen.a72==1)&(screen.a73==1)&(screen.a74==0),
    "a7new"
]=2

screen.loc[screen.scr_a7.astype(str).str.contains("5",na=False),"a7new"]=3
screen.loc[screen.a7.astype(str).str.contains("5",na=False),"a7new"]=3


# ---------------------------------------------------------
# DATE
# ---------------------------------------------------------

screen["recruitdate"] = np.where(
    screen["cohort"]==1,
    pd.to_datetime(screen["scr_a2"],errors="coerce"),
    pd.to_datetime(screen["a2"],errors="coerce")
)


# ---------------------------------------------------------
# DUPLICATES
# ---------------------------------------------------------

screen = detect_duplicates(screen,"a1")

screen = screen[
    ~((screen.a1=="DEM2-PP-0019") & (screen.a6=="KH-25"))
]

screen = screen[
    ~((screen.a1=="DEM1-PP-0073") &
      (screen.i3e=="2025-12-12T14:16:10.213+07:00"))
]


save_dta(screen,"Demele-screening-cohort1&2-cleaned.dta")


# ---------------------------------------------------------
# 2. LAB DATA
# ---------------------------------------------------------

print("Loading lab data...")

elisa_c1 = load_excel(
    lab_elisa_c1,
    "Sheet 1 - C1_LabResult_Cohort1_",
    "A:BC"
)

elisa_c2 = load_excel(
    lab_elisa_c2,
    "Sheet 1 - C4_LabResult_Cohort2_",
    "A:CA"
)

pcr_c2 = load_excel(
    lab_pcr_c2,
    "Sheet 1 - C5_LabResult_Cohort2_",
    "A:DI"
)

genx_c2 = load_excel(
    lab_genx_c2,
    "Sheet 1 - C6_LabResult_Cohort2_",
    "A:AG"
)

elisa_c1 = fix_ids(elisa_c1)
elisa_c2 = fix_ids(elisa_c2)
pcr_c2 = fix_ids(pcr_c2)
genx_c2 = fix_ids(genx_c2)


# ---------------------------------------------------------
# DATE VARIABLES
# ---------------------------------------------------------

elisa_c1["testdate"]=pd.to_datetime(elisa_c1["a1_5"],errors="coerce")
elisa_c2["testdate"]=pd.to_datetime(elisa_c2["a1_5"],errors="coerce")


# ---------------------------------------------------------
# MERGE LAB DATA
# ---------------------------------------------------------

lab = elisa_c2.merge(pcr_c2,on="a1",how="left")
lab = lab.merge(genx_c2,on="a1",how="left")

lab = pd.concat([lab, elisa_c1], ignore_index=True)


save_dta(lab,"Demele-D0-lab-all-cohort1&2.dta")


# ---------------------------------------------------------
# 3. MERGE SCREENING + LAB
# ---------------------------------------------------------

print("Merging screening and lab...")

df = lab.merge(screen,on="a1",how="left")


# ---------------------------------------------------------
# TEST AND COLLECTION DATE
# ---------------------------------------------------------

df["collectdate"]=pd.to_datetime(df["a2"],errors="coerce")
df["testdate"]=pd.to_datetime(df["a1_5"],errors="coerce")


# ---------------------------------------------------------
# BASELINE IgG VARIABLES
# ---------------------------------------------------------

df["iggd0serumjev"]=np.select(
    [
        (df.b5==1)|(df.d0_elic2_c2_5==1),
        (df.b5==2)|(df.d0_elic2_c2_5==2),
        (df.b5==3)|(df.d0_elic2_c2_5==3)
    ],
    [1,2,3],
    default=np.nan
)

df["iggd0serumden"]=np.select(
    [
        (df.c5==1)|(df.d0_elic2_c4_5==1),
        (df.c5==2)|(df.d0_elic2_c4_5==2),
        (df.c5==3)|(df.d0_elic2_c4_5==3)
    ],
    [1,2,3],
    default=np.nan
)


# ---------------------------------------------------------
# IgM VARIABLES
# ---------------------------------------------------------

df["igmd0serumden"]=df["d0_elic2_c3_5"]
df["igmd0serumjev"]=df["d0_elic2_c1_5"]

df["igmd0csfden"]=df["d0_elic2_b3_5"]
df["igmd0csfjev"]=df["d0_elic2_b1_5"]


# ---------------------------------------------------------
# SAVE FINAL DATASET
# ---------------------------------------------------------

save_dta(df,"Demele-D0-screening&lab-cohort1&2-cleaned.dta")

print("Pipeline completed successfully.")